In [1]:
from getdist import loadMCSamples
import getdist.plots as plots
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from scipy.stats import chi2, norm
from cobaya.yaml import yaml_load

import sys
sys.path.append('/home/theppawan/cosmo-research/quintom-project/scripts')
sys.path.append('/home/theppawan/cosmo-research/quintom-project/likelihood')
from compute_dic import compute_dic
chain_dir = Path(r'/home/theppawan/cosmo-research/quintom-project/chains')

In [2]:
lcdm = {'DESI': str(chain_dir / 'lcdm/DESI/lcdm_DESI'),
        'DESI+BBN': str(chain_dir / 'lcdm/DESI+BBN/lcdm_DESI+BBN'),
        'DESI+BBN+SNIa': str(chain_dir / 'lcdm/DESI+BBN+SNIa/lcdm_DESI+BBN+SNIa'),
        'DESI+BBN+SNIa+SH0ES': str(chain_dir / 'lcdm/DESI+BBN+SNIa+SH0ES/lcdm_DESI+BBN+SNIa+SH0ES'),
        'DESI+CMB': str(chain_dir / 'lcdm/DESI+CMB/lcdm_DESI+CMB'),
        'DESI+CMB+SNIa': str(chain_dir / 'lcdm/DESI+CMB+SNIa/lcdm_DESI+CMB+SNIa'),
        'DESI+CMB+SNIa+SH0ES': str(chain_dir / 'lcdm/DESI+CMB+SNIa+SH0ES/lcdm_DESI+CMB+SNIa+SH0ES')
        }
quintom = {'DESI': str(chain_dir / 'quintom/DESI/quintom_DESI'),
           'DESI+BBN': str(chain_dir / 'quintom/DESI+BBN/quintom_DESI+BBN'),
           'DESI+BBN+SNIa': str(chain_dir / 'quintom/DESI+BBN+SNIa/quintom_DESI+BBN+SNIa'),
           'DESI+BBN+SNIa+SH0ES': str(chain_dir / 'quintom/DESI+BBN+SNIa+SH0ES/quintom_DESI+BBN+SNIa+SH0ES'),
           'DESI+CMB': str(chain_dir / 'quintom/DESI+CMB/quintom_DESI+CMB'),
           'DESI+CMB+SNIa': str(chain_dir / 'quintom/DESI+CMB+SNIa/quintom_DESI+CMB+SNIa'),
           'DESI+CMB+SNIa+SH0ES': str(chain_dir / 'quintom/DESI+CMB+SNIa+SH0ES/quintom_DESI+CMB+SNIa+SH0ES')
        }

### 1. Difference in Maximum Posterior Fit

In [3]:
def compute_chi_sq_MAP(chain_path, burnin):
    sample = loadMCSamples(chain_path, settings={'ignore_rows': burnin})
    yaml_file = f"{chain_path}.updated.yaml"   
    with open(yaml_file, 'r') as f:
        info = yaml_load(f)
    likelihood_dict = info.get('likelihood', {})
    expected_chi2_cols = [f"chi2__{name}" for name in likelihood_dict.keys()]
    best_fit_params = sample.getParamBestFitDict(best_sample=False)
    chi2_cols = [col for col in expected_chi2_cols if col in best_fit_params.keys()]
    
    chi_sq_map = 0
    for chi2_i in chi2_cols:
        chi_sq_map += best_fit_params[chi2_i]
    
    return chi_sq_map

def compute_delta_chi_sq_MAP(chain_path_1, chain_path_2, burnin):
    chi_sq1 = compute_chi_sq_MAP(chain_path_1, burnin=burnin)
    chi_sq2 = compute_chi_sq_MAP(chain_path_2, burnin=burnin)

    return chi_sq1 - chi_sq2

### 2. Significance (Statistical Preference)

In [4]:
def compute_significance(chain_path_1, chain_path_2, burnin):

    delta_chi2_map = compute_delta_chi_sq_MAP(chain_path_1, chain_path_2, burnin)
    if delta_chi2_map >= 0:
            return 0.0

    delta_dof = 2
    improvement = abs(delta_chi2_map)
    p_value = chi2.sf(improvement, delta_dof)
    sigma = norm.ppf(1.0 - (p_value / 2.0))
    
    return sigma

### 3. $\Delta(\text{DIC})$ (Difference in Deviance Information Criterion)

In [5]:
def compute_delta_dic(chain_path_1, chain_path_2, burnin):
    dic1, _ = compute_dic(chain_path_1, burnin_fraction=burnin)
    dic2, _ = compute_dic(chain_path_2, burnin_fraction=burnin)

    return dic1 - dic2

### Analysis

In [6]:
dataset = ['DESI', 
           'DESI+BBN', 'DESI+BBN+SNIa', 'DESI+BBN+SNIa+SH0ES', 
           'DESI+CMB', 'DESI+CMB+SNIa', 'DESI+CMB+SNIa+SH0ES']
for ds in dataset:
    burn_in = 0.3
    delta_chi2 = compute_delta_chi_sq_MAP(quintom[ds], lcdm[ds], burn_in)
    sign = compute_significance(quintom[ds], lcdm[ds], burn_in)
    delta_dic = compute_delta_dic(quintom[ds], lcdm[ds], burn_in)
    table = ds + f'& ${delta_chi2:.2f}$' + rf'& ${sign:.2f}\sigma$' + f'& ${delta_dic:.2f}$' + "\\\\"
    print(table)

DESI& $-0.20$& $0.12\sigma$& $0.74$\\
DESI+BBN& $-14.68$& $3.41\sigma$& $-9.74$\\
DESI+BBN+SNIa& $-17.64$& $3.79\sigma$& $-11.82$\\
DESI+BBN+SNIa+SH0ES& $-17.03$& $3.72\sigma$& $-10.82$\\
DESI+CMB& $-6.51$& $2.07\sigma$& $-3.16$\\
DESI+CMB+SNIa& $-7.69$& $2.30\sigma$& $-3.39$\\
DESI+CMB+SNIa+SH0ES& $0.01$& $0.00\sigma$& $3.06$\\


In [10]:
delta_dof = 2
improvement = abs(-8)
p_value = chi2.sf(improvement, delta_dof)
sigma = norm.ppf(1.0 - (p_value / 2.0))
sigma

np.float64(2.3591745411622598)